# 05 — Diagnostics & Validation

Aggregate out-of-sample predictions and produce the Phase 1 PASS/FAIL decision.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
from google.colab import runtime
from yenibot.experiments import latest_experiment_run, write_experiment_diagnostics

REPORT_DIR = f'{DRIVE_BASE}/reports'

try:
    run_dir = latest_experiment_run(CHECKPT_DIR)
    print('Latest experiment run:', run_dir)
    result = write_experiment_diagnostics(
        checkpoint_dir=CHECKPT_DIR,
        config=cfg,
        output_dir=REPORT_DIR,
    )
    display(result['comparison'])
    print('Decision:', result['decision'])
    print('Diagnostic zips:')
    for zip_path in result['zip_paths']:
        print(zip_path)
finally:
    runtime.unassign()
